## Preparation

In [36]:
import os
from pytube import YouTube
import librosa
import json
from tqdm import tqdm
import pandas

In [2]:
with open('./MSRVTT_data.json','r') as j:
    data = json.load(j)

In [3]:
data.keys()

dict_keys(['info', 'videos', 'sentences'])

In [4]:
meta_info = data['info']
meta_video = data['videos']
meta_sentence = data['sentences']

In [32]:
meta_video[0].keys(), meta_sentence[0].keys()

(dict_keys(['category', 'url', 'video_id', 'start time', 'end time', 'split', 'id']),
 dict_keys(['caption', 'video_id', 'sen_id']))

In [35]:
meta_video[0], meta_sentence[0]

({'category': 9,
  'url': 'https://www.youtube.com/watch?v=9lZi22qLlEo',
  'video_id': 'video0',
  'start time': 137.72,
  'end time': 149.44,
  'split': 'train',
  'id': 0},
 {'caption': 'a cartoon animals runs through an ice cave in a video game',
  'video_id': 'video2960',
  'sen_id': 0})

In [39]:
with open('./MSRVTT_JSFUSION_test.csv', 'r') as f:
    test_original = pandas.read_csv(f)

In [45]:
test_original.keys()

Index(['key', 'vid_key', 'video_id', 'sentence'], dtype='object')

In [51]:
test_original['key'][0], test_original['vid_key'][0], test_original['video_id'][0], test_original['sentence'][0]

('ret0', 'msr9770', 'video9770', 'a person is connecting something to system')

In [112]:
with open('./MSRVTT_train.9k.csv', 'r') as tr:
    train_9k = pandas.read_csv(tr)

In [113]:
train_9k.keys()


Index(['video_id'], dtype='object')

In [114]:
def find_sentence(video_id):
    for i in range(len(meta_sentence)):
        sent =[]
        if meta_sentence[i]['video_id'] != video_id:
            continue
        else: 
            sent.append(meta_sentence[i]['caption'])
        if len(sent) > 0:
            return sent[0]

## Dataset Construction for 4-fold Cross-Validation

### Dataset Split 

In [121]:
for fold in range(4):
    val_fold = {'key':[], 'vid_key':[], 'video_id':[], 'sentence':[]}
    for i in range(2250):
        fold_i = 2250*fold + i
        vid = train_9k['video_id'][fold_i]
        val_fold['key'].append('ret'+str(fold_i))
        val_fold['vid_key'].append('msr'+vid[5:])
        val_fold['video_id'].append(vid)
        val_fold['sentence'].append(find_sentence(vid))

    df_val_fold = pandas.DataFrame(val_fold)
    df_val_fold.to_csv('./MSRVTT_val_fold_'+str(fold+1)+'.csv', index=False)


In [122]:
# for i in range(2250):
#     vid = train_9k['video_id'][i]
#     train_fold['video_id'].append(vid)
#     val_fold['key'].append('ret'+str(i))
#     val_fold['vid_key'].append('msr'+vid[5:])
#     val_fold['video_id'].append(vid)
#     val_fold['sentence'].append(find_sentence(vid))


In [123]:
# df_train_fold = pandas.DataFrame(train_fold)
# df_train_fold.to_csv('./MSRVTT_train_fold1.csv', index=False)
# df_val_fold = pandas.DataFrame(val_fold)
# df_val_fold.to_csv('./MSRVTT_val_fold1.csv', index=False)


In [148]:
subset = train_9k.iloc[list(range(2250,9000))]
subset.to_csv('./MSRVTT_train_fold_1.csv', index=False)

subset = train_9k.iloc[list(range(0,2250))+ list(range(4500, 9000))]
subset.to_csv('./MSRVTT_train_fold_2.csv', index=False)

subset = train_9k.iloc[list(range(0,4500))+ list(range(6750, 9000))]
subset.to_csv('./MSRVTT_train_fold_3.csv', index=False)

subset = train_9k.iloc[list(range(0, 6750))]
subset.to_csv('./MSRVTT_train_fold_4.csv', index=False)

In [145]:
train_9k['video_id'][list(range(0,10))+  list(range(20,30))]

TypeError: unsupported operand type(s) for -: 'list' and 'list'

In [130]:
from sklearn.model_selection import KFold



In [131]:
kf = KFold(n_splits=4, shuffle=False)


In [132]:
train_1fold =kf.split(train_9k)

In [134]:
train_1fold[0]

TypeError: 'generator' object is not subscriptable